# Notebook 2 — Who Matters?
**YTS+ DSEB 2026 · Project B · Indian Railways**

---

## The question we are building toward

*Which station, if it shut down, would cause the most disruption — and is it the one everyone has heard of?*

In network science there are three different answers to "who matters":

| Metric | Question | What it counts |
|---|---|---|
| **Degree** | Who is most connected? | Number of direct track neighbours |
| **Weighted degree** | Who handles the most traffic? | Total trains flowing through direct connections |
| **Betweenness** | Who controls the flow? | Fraction of shortest paths passing through |

The first two lists look similar. The third looks completely different. By the end of this notebook you will know why — and you will have an answer to the big question.

**Keep your sealed predictions from Session 1 sealed. You will open them at the very end.**

---
## Setup

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'

print('Ready.')

In [ ]:
# Load stations and the physical track network (built in Session 1)
stations = pd.read_csv(DATA + 'stations_clean.csv')
track_edges = pd.read_csv(DATA + 'track_edges.csv')

# Build the graph
G = nx.Graph()
for _, row in track_edges.iterrows():
    G.add_edge(row['station_a'], row['station_b'], weight=row['num_trains'])

print(f'Track network: {G.number_of_nodes():,} stations, {G.number_of_edges():,} edges')

# Coordinate lookup
coords = stations.set_index('code')[['lat', 'lon']].to_dict('index')
name_lookup = stations.set_index('code')['name'].to_dict()
state_lookup = stations.set_index('code')['state'].to_dict()

print('Data loaded.')

---
## Part 1 — Degree: who is most connected?

**Degree** = the number of stations directly linked to this one by track.

A station with degree 5 has 5 track connections going out from it.

**Before you compute: predict the top 5 stations by degree.**

Think about which cities are the biggest railway hubs in India.

*Your prediction (write before running the next cell):*

In [ ]:
# Compute degree for every station
degree = dict(G.degree())

# Build a ranked dataframe
degree_df = pd.DataFrame([
    {'code': code,
     'name': name_lookup.get(code, code),
     'state': state_lookup.get(code, ''),
     'degree': deg}
    for code, deg in degree.items()
]).sort_values('degree', ascending=False).reset_index(drop=True)

degree_df['degree_rank'] = degree_df.index + 1

print('Top 20 stations by degree (number of direct track connections):')
print()
print(f'{"Rank":<6} {"Code":<8} {"Name":<35} {"State":<20} {"Degree":>7}')
print('-' * 78)
for _, row in degree_df.head(20).iterrows():
    print(f"{int(row['degree_rank']):<6} {row['code']:<8} {str(row['name']):<35} {str(row['state']):<20} {int(row['degree']):>7}")

In [ ]:
# Weighted degree: for each station, sum the num_trains on all its track connections
# This measures actual train traffic volume flowing through this station

weighted_deg = {
    node: sum(G[node][nbr]['weight'] for nbr in G.neighbors(node))
    for node in G.nodes()
}

wd_df = pd.DataFrame([
    {'code': code,
     'name': name_lookup.get(code, code),
     'state': state_lookup.get(code, ''),
     'weighted_degree': wd}
    for code, wd in weighted_deg.items()
]).sort_values('weighted_degree', ascending=False).reset_index(drop=True)

wd_df['wd_rank'] = wd_df.index + 1

print('Top 20 stations by weighted degree (total trains on direct track connections):')
print()
print(f'{"Rank":<6} {"Code":<8} {"Name":<35} {"State":<20} {"Train flows":>12}')
print('-' * 83)
for _, row in wd_df.head(20).iterrows():
    print(f"{int(row['wd_rank']):<6} {row['code']:<8} {str(row['name']):<35} {str(row['state']):<20} {int(row['weighted_degree']):>12,}")

In [ ]:
# Distribution of degrees
deg_values = list(degree.values())

print(f'Most stations have very few connections:')
print(f'  Degree 1 (dead end):    {deg_values.count(1):,}')
print(f'  Degree 2 (pass-through):{deg_values.count(2):,}')
print(f'  Degree 3 (junction):    {deg_values.count(3):,}')
print(f'  Degree 4+:              {sum(1 for d in deg_values if d >= 4):,}')
print(f'  Degree 8+:              {sum(1 for d in deg_values if d >= 8):,}')
print(f'  Max degree:             {max(deg_values)}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(deg_values, bins=range(1, max(deg_values)+2), color='steelblue', edgecolor='white')
ax.set_xlabel('Degree (number of track connections)')
ax.set_ylabel('Number of stations')
ax.set_title('Most Indian railway stations are pass-throughs or dead ends')
ax.set_xlim(0, 20)
plt.tight_layout()
plt.savefig('images/degree_distribution.png', dpi=150)
plt.show()

In [ ]:
# Map: dot size proportional to degree
# Only plot stations in coords
plot_codes = [c for c in degree_df['code'] if c in coords]
lons = [coords[c]['lon'] for c in plot_codes]
lats = [coords[c]['lat'] for c in plot_codes]
degs = [degree.get(c, 1) for c in plot_codes]

fig, ax = plt.subplots(figsize=(8, 10))
sc = ax.scatter(lons, lats, s=[d**1.8 for d in degs],
                c=degs, cmap='YlOrRd', alpha=0.6, linewidths=0)
plt.colorbar(sc, ax=ax, label='Degree')

# Label top 10 by degree
for _, row in degree_df.head(10).iterrows():
    c = row['code']
    if c in coords:
        ax.annotate(row['name'][:15],
                    (coords[c]['lon'], coords[c]['lat']),
                    fontsize=6.5, color='black',
                    xytext=(3, 2), textcoords='offset points')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('India — station size = degree (number of track connections)')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
plt.tight_layout()
plt.savefig('images/degree_map.png', dpi=150)
plt.show()

**Write:** Which cities dominate the top 20 by degree? Were your predictions correct? What do the high-degree stations have in common — are they in specific regions of India?

*Your observation:*

---
## Part 2 — Betweenness: who controls the flow?

**Betweenness centrality** asks a different question:

> *If every pair of stations wanted to send a train between them via the shortest path, what fraction of those paths would pass through station X?*

A station can have **low degree** (few direct neighbours) but **high betweenness** if it sits at a choke point that all paths must use.

**Computing betweenness on 8,000+ stations takes hours.** We pre-computed it for you. You are loading the result.

In [ ]:
# Load pre-computed betweenness
bc = pd.read_csv(DATA + 'betweenness_track.csv')

# Add name and state
bc['name'] = bc['code'].map(name_lookup).fillna(bc['code'])
bc['state'] = bc['code'].map(state_lookup).fillna('')

bc = bc.sort_values('betweenness_track', ascending=False).reset_index(drop=True)
bc['betweenness_rank'] = bc.index + 1

print('Top 25 stations by betweenness centrality (physical track network):')
print()
print(f'{"Rank":<6} {"Code":<8} {"Name":<35} {"State":<20} {"Betweenness":>12}')
print('-' * 83)
for _, row in bc.head(25).iterrows():
    print(f"{int(row['betweenness_rank']):<6} {row['code']:<8} {str(row['name']):<35} {str(row['state']):<20} {row['betweenness_track']:>12.4f}")

**Write:** Compare this list to the degree top 20.
- Which stations appear in BOTH lists?
- Which stations appear only in the betweenness list (high betweenness, lower degree)?
- Where in India are the top betweenness stations concentrated?

*Your observation:*

---
## Part 3 — The gap between degree and betweenness

If degree and betweenness measured the same thing, every station's rank on one would equal its rank on the other.

They do not. Let us see where the biggest gaps are.

In [ ]:
# Three-way comparison: degree rank, weighted degree rank, betweenness rank
# This is the core finding of the notebook

# Build a single merged table
merged = degree_df[['code', 'name', 'state', 'degree', 'degree_rank']].merge(
    wd_df[['code', 'weighted_degree', 'wd_rank']],
    on='code', how='inner'
).merge(
    bc[['code', 'betweenness_track', 'betweenness_rank']],
    on='code', how='inner'
)

# Show the top 20 by betweenness — the stations that matter structurally
print('TOP 20 BY BETWEENNESS — compared to degree and traffic ranks')
print('(Read: is this station also highly ranked by traffic or connections?)')
print()
print(f'{"BC Rank":<8} {"Code":<8} {"Name":<28} {"Deg Rank":>9} {"Traffic Rank":>13}')
print('-' * 70)
for _, row in merged.sort_values('betweenness_rank').head(20).iterrows():
    print(f"#{int(row['betweenness_rank']):<7} {row['code']:<8} {str(row['name']):<28} "
          f"#{int(row['degree_rank']):<8} #{int(row['wd_rank']):<12}")

print()
print('TOP 20 BY WEIGHTED DEGREE — compared to betweenness rank')
print('(Read: do the busiest stations also score high on betweenness?)')
print()
print(f'{"Traf Rank":<10} {"Code":<8} {"Name":<28} {"Deg Rank":>9} {"BC Rank":>8}')
print('-' * 67)
for _, row in merged.sort_values('wd_rank').head(20).iterrows():
    print(f"#{int(row['wd_rank']):<9} {row['code']:<8} {str(row['name']):<28} "
          f"#{int(row['degree_rank']):<8} #{int(row['betweenness_rank']):<8}")

In [ ]:
# Scatter: weighted degree rank vs betweenness rank
# If traffic volume predicted structural importance, all points would be on the diagonal

plot_df = merged[merged['betweenness_rank'] <= 300].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (x_col, x_label) in zip(axes, [
    ('degree_rank', 'Degree rank'),
    ('wd_rank',     'Weighted degree rank (traffic)'),
]):
    ax.scatter(plot_df[x_col], plot_df['betweenness_rank'],
               s=8, alpha=0.4, color='steelblue')

    # diagonal = perfect agreement
    ax.plot([0, 300], [0, 300], 'k--', linewidth=0.8, alpha=0.4, label='Perfect agreement')

    # Label key stations
    for code in ['KIUL', 'NDLS', 'MB', 'PNBE', 'BCT', 'LKR']:
        row = merged[merged['code'] == code]
        if len(row) == 0:
            continue
        r = row.iloc[0]
        ax.annotate(f"{code}",
                    (r[x_col], r['betweenness_rank']),
                    fontsize=8, color='tomato',
                    xytext=(5, 2), textcoords='offset points')
        ax.scatter([r[x_col]], [r['betweenness_rank']],
                   s=50, color='tomato', zorder=5)

    ax.set_xlabel(f'{x_label} (1 = highest)')
    ax.set_ylabel('Betweenness rank (1 = most critical)')
    ax.set_title(f'{x_label}\nvs Betweenness rank')
    ax.legend(fontsize=8)
    ax.invert_xaxis()
    ax.invert_yaxis()

plt.suptitle('Traffic measures (left/right) vs Betweenness — top 300 by betweenness', y=1.02)
plt.tight_layout()
plt.savefig('images/traffic_vs_betweenness.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key stations — all three rankings:')
print(f'{"Code":<8} {"Name":<28} {"Degree":>8} {"Traffic":>9} {"Betweenness":>12}')
print('-' * 68)
for code in ['NDLS', 'BCT', 'MAS', 'KIUL', 'MB', 'LKR', 'PNBE']:
    row = merged[merged['code'] == code]
    if len(row) == 0:
        continue
    r = row.iloc[0]
    print(f"{r['code']:<8} {str(r['name']):<28} #{int(r['degree_rank']):>7} #{int(r['wd_rank']):>8} #{int(r['betweenness_rank']):>11}")

**Write:** Look at the two scatter plots and the key stations table.

- New Delhi and Mumbai are near the top for both degree and traffic — they are genuinely busy. But where do they rank by betweenness?
- Kiul Junction is low on degree and low on traffic — almost no express trains stop there. But where does it rank by betweenness?
- What does this mean? Can a station be structurally critical **without** being a busy station?
- Why does train traffic NOT predict which station would cause the most disruption if it shut down?

*Your answer:*

---
## Part 4 — Map colored by betweenness

Let us see WHERE in India the high-betweenness stations are.

In [ ]:
# Build betweenness lookup
bc_lookup = bc.set_index('code')['betweenness_track'].to_dict()

plot_codes = [c for c in G.nodes() if c in coords]
lons = [coords[c]['lon'] for c in plot_codes]
lats = [coords[c]['lat'] for c in plot_codes]
bcs  = [bc_lookup.get(c, 0) for c in plot_codes]

fig, ax = plt.subplots(figsize=(9, 11))

# All stations as small grey dots
ax.scatter(lons, lats, s=1.5, color='lightgrey', alpha=0.5, zorder=1)

# Overlay betweenness with color
# Only show stations with betweenness > 0
top_mask = [b > 0 for b in bcs]
lons_top = [l for l, m in zip(lons, top_mask) if m]
lats_top = [l for l, m in zip(lats, top_mask) if m]
bcs_top  = [b for b, m in zip(bcs, top_mask) if m]

sc = ax.scatter(lons_top, lats_top,
                s=[max(4, b * 15000) for b in bcs_top],
                c=bcs_top, cmap='hot_r',
                alpha=0.75, zorder=2, linewidths=0)
plt.colorbar(sc, ax=ax, label='Betweenness centrality')

# Label top 10
for _, row in bc.head(10).iterrows():
    c = row['code']
    if c in coords:
        ax.annotate(str(row['name'])[:15],
                    (coords[c]['lon'], coords[c]['lat']),
                    fontsize=6.5, color='black', fontweight='bold',
                    xytext=(3, 3), textcoords='offset points', zorder=5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('India — betweenness centrality of railway stations')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
plt.tight_layout()
plt.savefig('images/betweenness_map_india.png', dpi=150)
plt.show()

In [ ]:
# Zoom into Bihar — the hot zone
fig, ax = plt.subplots(figsize=(10, 8))

# Grey background of all stations in the region
region_codes = [c for c in plot_codes
                if 24.5 < coords[c]['lat'] < 27.5 and 83 < coords[c]['lon'] < 88]
r_lons = [coords[c]['lon'] for c in region_codes]
r_lats = [coords[c]['lat'] for c in region_codes]
r_bcs  = [bc_lookup.get(c, 0) for c in region_codes]

ax.scatter(r_lons, r_lats, s=4, color='lightgrey', alpha=0.6, zorder=1)

# Color overlay
top_r = [(l, la, b) for l, la, b in zip(r_lons, r_lats, r_bcs) if b > 0]
if top_r:
    tl, tla, tb = zip(*top_r)
    sc2 = ax.scatter(tl, tla,
                     s=[max(8, b * 12000) for b in tb],
                     c=tb, cmap='hot_r', alpha=0.85, zorder=2, linewidths=0)
    plt.colorbar(sc2, ax=ax, label='Betweenness centrality')

# Label top stations in this region
region_top = bc[bc['code'].isin(region_codes)].head(12)
for _, row in region_top.iterrows():
    c = row['code']
    if c in coords:
        ax.annotate(f"{row['name']} (#{int(row['betweenness_rank'])})",
                    (coords[c]['lon'], coords[c]['lat']),
                    fontsize=8, color='black', fontweight='bold',
                    xytext=(4, 3), textcoords='offset points')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Bihar/UP zoom — the betweenness hot zone')
ax.set_xlim(83, 88)
ax.set_ylim(24.5, 27.5)
plt.tight_layout()
plt.savefig('images/betweenness_map_bihar.png', dpi=150)
plt.show()

**Write:** Look at the Bihar zoom. Almost all of the top betweenness stations sit in a tight band. 

- What geographic feature runs through this region?
- Why would that feature force all railway traffic to pass through very few points?
- Look at the national map: is the Deccan Plateau (south-central India) also a hot zone? Why or why not?

*Your answer:*

---
## Part 5 — What if we remove a station?

The real test of "who matters" is: if this station shuts down, how many other stations lose their connection to the rest of India?

We will compare:
- Removing **Kiul Junction** (KIUL) — high betweenness, low degree
- Removing **New Delhi** (NDLS) — high degree, lower betweenness
- Removing a random small station

**Before you run: predict.** If Kiul Junction shut down, how many stations do you think would lose all connection to the rest of India? Would New Delhi's closure affect more or fewer?

*Your prediction:*

In [ ]:
def stations_disconnected(G, node_to_remove):
    """How many stations lose connection to the largest component when this node is removed?"""
    G2 = G.copy()
    G2.remove_node(node_to_remove)
    components = list(nx.connected_components(G2))
    # Largest component = 'rest of India'
    main = max(components, key=len)
    stranded = sum(len(c) for c in components if c != main)
    n_components = len(components)
    return stranded, n_components


# Baseline
baseline_components = nx.number_connected_components(G)
print(f'Baseline: {G.number_of_nodes():,} stations, {baseline_components} connected component(s)\n')

# Test several stations
test_stations = [
    ('KIUL', 'Kiul Junction'),
    ('NDLS', 'New Delhi'),
    ('MB',   'Moradabad'),
    ('BCT',  'Mumbai Central'),
    ('MAS',  'Chennai Central'),
    ('PNBE', 'Patna Junction'),
]

print(f'{"Station":<30} {"Stranded stations":>18} {"Components after":>17}')
print('-' * 67)

results = []
for code, label in test_stations:
    if code not in G:
        print(f'{label} ({code}) — not in graph')
        continue
    stranded, n_comp = stations_disconnected(G, code)
    results.append((code, label, stranded, n_comp))
    row_label = f'{label} ({code})'
    print(f'{row_label:<30} {stranded:>18,} {n_comp:>17}')

In [ ]:
# Bar chart of the results
if results:
    labels_plot = [f"{r[1]}\n({r[0]})" for r in results]
    stranded_vals = [r[2] for r in results]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels_plot, stranded_vals, color='steelblue', edgecolor='white')

    # Highlight the top bar
    max_idx = stranded_vals.index(max(stranded_vals))
    bars[max_idx].set_color('tomato')

    ax.set_ylabel('Stations stranded (disconnected from rest of India)')
    ax.set_title('Which station\'s closure causes the most disruption?')
    for bar, val in zip(bars, stranded_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('images/removal_impact.png', dpi=150)
    plt.show()

**Write:** 
- Was your prediction right?
- New Delhi is the most famous railway station in India. Why does its closure strand fewer stations than a station many people have never heard of?
- What does this tell you about the difference between *popularity* and *structural importance* in a network?

*Your answer:*

---
## Part 6 — One more look: where did Kiul come from?

In Notebook 1 we built the stop network (consecutive train stops) and then the track network (physical rails). Let us check how Kiul's rank changed.

In [ ]:
# Load betweenness on the stop network (if available)
import os

if os.path.exists(DATA + 'betweenness_scores.csv'):
    bc_stop = pd.read_csv(DATA + 'betweenness_scores.csv')
    # The column is called 'betweenness' in the stop network file
    bc_stop = bc_stop.sort_values('betweenness', ascending=False).reset_index(drop=True)
    bc_stop['stop_rank'] = bc_stop.index + 1

    # Merge with track betweenness
    compare = bc_stop[['code', 'betweenness', 'stop_rank']].merge(
        bc[['code', 'name', 'betweenness_track', 'betweenness_rank']],
        on='code', how='inner'
    )
    compare['rank_change'] = compare['stop_rank'] - compare['betweenness_rank']

    print('Stations that ROSE in betweenness rank: stop → track network')
    print('(These are the true physical hinges the express network was hiding)')
    print()
    print(f'{"Code":<8} {"Name":<30} {"Stop rank":>10} {"Track rank":>11} {"Rise":>6}')
    print('-' * 68)
    for _, row in compare.nlargest(15, 'rank_change').iterrows():
        print(f"{row['code']:<8} {str(row['name']):<30} {int(row['stop_rank']):>10} {int(row['betweenness_rank']):>11} {int(row['rank_change']):>+6}")

    print()
    print('Stations that FELL — inflated by express train connections:')
    print(f'{"Code":<8} {"Name":<30} {"Stop rank":>10} {"Track rank":>11} {"Drop":>6}')
    print('-' * 68)
    for _, row in compare.nsmallest(15, 'rank_change').iterrows():
        print(f"{row['code']:<8} {str(row['name']):<30} {int(row['stop_rank']):>10} {int(row['betweenness_rank']):>11} {int(row['rank_change']):>+6}")
else:
    print('betweenness_scores.csv not found — skipping stop/track comparison.')
    print('This is fine — the track betweenness results above are what matter.')

**Write:** Why do stations that are major express train stops fall in rank when we move from the stop network to the physical track network? What does that tell us about what express trains show us vs what they hide?

*Your answer:*

---
## Open the sealed predictions

In Session 1 you wrote down the 5 stations you thought were most critical to India's railway network.

Open the paper now.

**Write:**
- Which of your predictions were correct?
- Which were wrong — and why?
- What does it tell you that most people would predict New Delhi or Mumbai — not Kiul Junction or Moradabad?

*Your reflection:*

---
## What to bring to the next session

- Your answer: why does Kiul Junction have such high betweenness?
- A geographic hypothesis: what physical feature forces traffic through the Bihar cluster?
- One question the data raised that you cannot answer yet

**Next:** Notebook 3 — Why Bihar? We measure how far trains must detour around geographic barriers, map those barriers to rivers, and ask the colonial question: when were the Bihar cluster stations built?